In [ ]:
import sys
import torch
import numpy as np

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("NumPy:", np.__version__)

In [ ]:
from pathlib import Path
import json

import numpy as np
import torch

# 项目根目录
PROJECT_ROOT = Path.cwd().parent

text_feature_path = PROJECT_ROOT / "results" / "coco5000_text_features.pt"
metadata_path = PROJECT_ROOT / "data" / "coco5000_metadata.json"
caption_index_path = PROJECT_ROOT / "data" / "coco5000_caption_image_indices.json"

output_dir = PROJECT_ROOT / "data" / "text_index"
output_dir.mkdir(parents=True, exist_ok=True)

print("加载 text features...")
text_features = torch.load(
    text_feature_path,
    map_location="cpu",
    weights_only=True
).float()

print("text_features:", text_features.shape)

if text_features.ndim != 2 or text_features.shape[1] != 512:
    raise ValueError(
        f"Unexpected text feature shape: {text_features.shape}"
    )

text_numpy = text_features.numpy().astype(np.float32)
text_numpy.tofile(output_dir / "text_features.bin")

print("导出二进制向量完成")

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(caption_index_path, "r", encoding="utf-8") as f:
    caption_image_indices = json.load(f)

captions = []
for item in metadata:
    captions.extend(item["captions"])

print("captions:", len(captions))
print("caption_image_indices:", len(caption_image_indices))

if len(captions) != len(caption_image_indices):
    raise ValueError("描述数量和映射数量不一致")

if len(captions) != text_features.shape[0]:
    raise ValueError("描述数量和文字向量数量不一致")

with open(
    output_dir / "captions.txt",
    "w",
    encoding="utf-8"
) as f:
    for caption in captions:
        caption = caption.replace("\n", " ").strip()
        f.write(caption + "\n")

with open(
    output_dir / "caption_image_indices.txt",
    "w",
    encoding="utf-8"
) as f:
    for image_index in caption_image_indices:
        f.write(f"{image_index}\n")

index_meta = {
    "num_captions": len(captions),
    "feature_dim": int(text_features.shape[1]),
    "dtype": "float32",
    "features_file": "text_features.bin",
    "captions_file": "captions.txt",
    "caption_image_indices_file": "caption_image_indices.txt",
}

with open(
    output_dir / "index_meta.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(index_meta, f, ensure_ascii=False, indent=2)

print("导出目录:", output_dir)
print("导出完成")